In [84]:
#import libraries
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
#this is for making custom stop words
from sklearn.feature_extraction._stop_words import ENGLISH_STOP_WORDS
#train test split for training and testing duh
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.svm import SVC
import numpy as np

In [85]:
#read csv files, give their columns titles
drawings = pd.read_csv(r"C:\Users\gramos1\OneDrive - Reworld\DrawingNames.csv",sep=',',header=0,names=['Drawing Title', 'Category'])
#Reading only the labeled drawings
cutdrawings = drawings[drawings['Category'].notna()].copy()

In [86]:
code_pattern = r'^[A-Za-z0-9]+[-_\s]?'
cutdrawings['Drawing Title'] = cutdrawings['Drawing Title'].str.replace(code_pattern, '', regex=True)

In [87]:
custom_stop_words = list(ENGLISH_STOP_WORDS) + ['sys',
                                                'installation',
                                                'detail',
                                               'schem',
                                               'diagram',
                                               'dwg',
                                               'diag',
                                               'plans',
                                               'layout',
                                               'details',
                                               'arrgt',]

In [88]:


#vect to bring in tfidf vectorizer
#stop words is to filter noise, english here to filter out generic english

vect = TfidfVectorizer(
     #trying to get rid of english and numbers
                       stop_words= custom_stop_words,
    #filter out anything not alphabetic
                        token_pattern=r'(?u)\b[a-zA-Z]+\b',
     #put this 1-2 so i can get bigrams, might need to increase? 1-3?
                      ngram_range=(1,3),
     #I want features to be words i think
                    analyzer = 'word',
      #min_df to try and filter out irrelevant words? usually drawings are in batches so if it's
        #Not important enough to be in a batch then dont count it
                     min_df=1,)

#fit_transform applied to drawing title names,
#Learn vocabulary and idf, return document-term matrix.
X = vect.fit_transform(cutdrawings['Drawing Title'])

In [89]:
#output of feature names for transformation
#(texts turns into numerical features)
print(vect.get_feature_names_out()[:200])

['absorber' 'absorber bott' 'absorber bott cone' 'absorber brackets'
 'absorber chain' 'absorber chain tensioner' 'absorber cone'
 'absorber cone sub' 'absorber elev' 'absorber elev orientation'
 'absorber field' 'absorber field bolt' 'absorber girder'
 'absorber girder section' 'absorber lower' 'absorber lower shellsub'
 'absorber manway' 'absorber manway frame' 'absorber misc'
 'absorber misc clips' 'absorber riser' 'absorber riser scrubber'
 'absorber riser section' 'absorber scraper' 'absorber scraper ring'
 'absorber scrubber' 'absorber scrubber pre' 'absorber shell'
 'absorber shell manway' 'absorber shim' 'absorber shim packs'
 'absorber shop' 'absorber shop ass' 'absorber upper'
 'absorber upper shell' 'ac' 'ac dc' 'ac dc interlocks' 'ac power'
 'ac power cubicle' 'ac pwr' 'ac pwr analyzer' 'ac regulator'
 'ac regulator sh' 'ac schematic' 'acces' 'acces openings'
 'acces openings scrubber' 'access' 'access collector'
 'access collector elev' 'access collector plan' 'access door

In [90]:
#prints out matrix with feature values, many will be zero, presented in this way to save memory
#remember matrix is one row per document and one col per token
#value = tfidf score 0-1, and the score quantifies how important that word is to the title
print(X[:50])


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 816 stored elements and shape (50, 5007)>
  Coords	Values
  (0, 1012)	0.258754346231252
  (0, 2903)	0.258754346231252
  (0, 4917)	0.15006522468505384
  (0, 2534)	0.1576089364609427
  (0, 1196)	0.17156282138122056
  (0, 2473)	0.24427928575153213
  (0, 3972)	0.11229042209061599
  (0, 1013)	0.258754346231252
  (0, 2904)	0.258754346231252
  (0, 4922)	0.258754346231252
  (0, 2553)	0.23400906689817572
  (0, 1212)	0.258754346231252
  (0, 2474)	0.24427928575153213
  (0, 1014)	0.258754346231252
  (0, 2905)	0.258754346231252
  (0, 4923)	0.258754346231252
  (0, 2555)	0.258754346231252
  (0, 1213)	0.258754346231252
  (1, 2534)	0.1473539192845593
  (1, 1196)	0.1603998777081046
  (1, 2473)	0.22838495686722154
  (1, 3972)	0.10498410918006618
  (1, 2553)	0.21878298229690732
  (1, 2474)	0.22838495686722154
  (1, 879)	0.24191817992844258
  :	:
  (48, 3964)	0.2955032632764328
  (48, 2468)	0.2955032632764328
  (48, 4395)	0.2955032632764328
  (4

In [91]:
#time to split up data
y = cutdrawings['Category']
X_train, X_test, y_train,y_test = train_test_split(X,y,test_size = 0.33, random_state = 42)

In [92]:
# Initialize and train the classifier
clf = SVC(kernel='linear')
clf.fit(X_train, y_train)

,C,1.0
,kernel,'linear'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [93]:
#predict on test set
y_pred = clf.predict(X_test)

#evaluate performance
accuracy = accuracy_score(y_test,y_pred)
report = classification_report(y_test, y_pred)

print(f'Accuracy: {accuracy:.4f}')
print('Classification Report:')
print(report)

Accuracy: 0.8358
Classification Report:
                               precision    recall  f1-score   support

                  Ash Removal       0.91      0.88      0.89        24
               Auxilary Water       0.67      1.00      0.80         8
   Boiler and Steam Generator       0.94      0.89      0.91        18
    Boiler-Furnace Combustion       1.00      0.94      0.97        17
                CEMS/Baghouse       1.00      0.43      0.60         7
             Carbon Injection       0.98      0.98      0.98        41
        Closed Cooling System       1.00      0.44      0.62         9
            Condensate System       1.00      0.62      0.76        13
                        DENOX       1.00      0.75      0.86        16
       Digital Control System       1.00      0.86      0.93        29
      Electrical Distribution       0.49      1.00      0.66        42
                    Feedwater       1.00      0.43      0.60         7
              Fire Protection       

C:\Users\gramos1\AppData\Local\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\gramos1\AppData\Local\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\gramos1\AppData\Local\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is

In [94]:
def predict_category(text):

    text_vec = vect.transform([text])
    prediction = clf.predict(text_vec)
    return prediction[0]
    

In [95]:
drawings.head()

,Drawing Title,Category
0,BIC-0200 - INSTRMNTN & CONTROL DIAGS. SYMB. & ...,NaN
1,BIC-0201 - MAIN & DUMP STM SYS. INSTRMNTN & CT...,NaN
2,BIC-0202 - EXT. STM & AUX. STM SYS. INSTRN & C...,NaN
3,BIC-0203 - FEEDWATER SYS. INSTRMTN & CTL DIAGS...,NaN
4,BIC-0204 - CONDS & MKE - UP WATER SYS. INSTR &...,Condensate System


In [96]:
cutdrawings.head()

,Drawing Title,Category
4,0204 - CONDS & MKE - UP WATER SYS. INSTR & CTL...,Condensate System
5,0205 - CLSD LOOP COOLING H20 SYS INSTR & CTL D...,Closed Cooling System
6,0206 - FUEL OIL & EMGCY DIESEL GE. SYS INSTR &...,Fuel Oil
7,0207 - INSTR. & PLANT AIR SYS. INSTRMNTN & CTL...,Service Air
9,0209 - MAIN TURB L - OIL & CO2 SYS INSTRMNTN &...,Main Turbine


In [97]:
# Next step is to fix this so it doesnt take the cut down version and actually sorts through all 5000 entries

completetfidf = vect.transform(drawings['Drawing Title'])
completepredictions = clf.predict(completetfidf)
drawings['Prediction'] = completepredictions

In [98]:
drawings['Accurate?'] = np.where((drawings['Category'] == drawings['Prediction']),1,0)

In [99]:
drawingpredictions = pd.DataFrame(drawings[['Drawing Title','Category','Prediction','Accurate?']])
file_name = 'OutputPredictions.xlsx'
drawingpredictions.to_excel(file_name)